In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

# Visão Geral do Dataset

In [ ]:
df = pd.read_csv('../data/raw/mental_health.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

# Qualidade dos Dados

In [ ]:
print('Valores nulo de cada coluna:')
df.isnull().sum()

In [ ]:
df.nunique()

In [ ]:
print('Número de valores duplicados:', df.duplicated().sum())

# Estatísticas Descritivas

In [ ]:
df.describe()

# Distribuição das Variavéis

In [ ]:
plt.bar(
    df['Gender'].value_counts().index, 
    df['Gender'].value_counts().values,
    color=['pink', 'lightblue']
)
plt.title('Distribution of Gender')

In [ ]:
df['Occupation'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette('pastel'))
plt.title('Distribution of Occupation')

In [ ]:
df['Physical_Activity'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette('pastel'))
plt.title('Physical Activity Distribution')

In [ ]:
variaveis_numericas = df.select_dtypes(include=['int64', 'float64']).columns
variaveis_numericas = variaveis_numericas.drop('Person_ID')
print(f"Temos {len(variaveis_numericas)} variaveis numéricas: {', '.join(variaveis_numericas)}")

f, axes = plt.subplots(7, 2, figsize=(15, 10))

line = 0
column = 0

for i in variaveis_numericas:
    sns.histplot(x=i, data=df, ax=axes[line][column], kde=True)
    column += 1
    if column == 2:
        column = 0
        line += 1

In [ ]:
df[variaveis_numericas].corr()

In [ ]:
sns.heatmap(df[variaveis_numericas].corr(), annot=True, cmap='coolwarm')

# Analise da variavel alvo: Depressão

In [ ]:
df.groupby('Depression').size()

In [ ]:
df.Depression.value_counts().plot(kind='bar', color=['salmon', 'lightgreen'])
plt.title('Depression Distribution')
plt.xlabel('Depression')

# Relação de cada variável x variável alvo

In [ ]:
# Set figure size and layout
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.autolayout'] = True

In [ ]:
sns.histplot(df, x='Age', hue='Depression', palette='Set2', multiple="layer", kde=True)
plt.title('Age Distribution by Depression Status')
plt.xlabel('Age')
plt.ylabel('Frequency')

In [ ]:
sns.countplot(x='Gender', hue='Depression', data=df, palette='Set2')
plt.title('Depression Rate by Gender')
plt.xlabel('Gender')
plt.ylabel('Depression Rate')

In [ ]:
sns.catplot(x='Occupation', hue='Depression', data=df, kind='count', palette='Set2')
plt.title('Depression Rate by Occupation')
plt.xlabel('Occupation')

In [ ]:
sns.histplot(df, x='Daily_Screen_Time', hue='Depression', kde=True, palette='Set2')
plt.title('Screen Time Distribution by Depression Status')
plt.xlabel('Daily Screen Time')
plt.ylabel('Frequency')

In [ ]:
sns.histplot(df, x='Social_Media_Usage', hue='Depression', kde=True, palette='Set2')
plt.title('Social Media Usage Distribution by Depression Status')
plt.xlabel('Daily Social Media Usage (hours)')
plt.ylabel('Frequency')

In [ ]:
sns.histplot(df, x='Sleep_Hours', hue='Depression', kde=True, palette='Set2')
plt.title('Sleep Hours Distribution by Depression Status')
plt.xlabel('Sleep Hours')

In [ ]:
sns.kdeplot(x='Stress_Level', hue='Depression', data=df, palette='Set2', fill=True)
plt.title('Stress Level Distribution by Depression Status')
plt.xlabel('Stress Level (1-10)')
plt.ylabel('Frequency')

In [ ]:
sns.histplot(df, x='Work_Study_Hours', hue='Depression', kde=True, palette='Set2')
plt.title('Work/Study Hours Distribution by Depression Status')
plt.xlabel('Daily Work/Study Hours')
plt.ylabel('Frequency')

In [ ]:
sns.countplot(x='Physical_Activity', hue='Depression', data=df, palette='Set2')

In [ ]:
sns.histplot(df, x='Social_Interaction_Score', hue='Depression', kde=True, palette='Set2')
plt.title('Social Interaction Score Distribution by Depression Status')
plt.xlabel('Daily Social Interaction Score (1-10)')
plt.ylabel('Frequency')

In [ ]:
sns.countplot(df, x='Caffeine_Intake', hue='Depression', palette='Set2')
plt.title('Caffeine Intake Distribution by Depression Status')
plt.xlabel('Daily Caffeine Intake (mg)')
plt.ylabel('Frequency')

# Avaliar outliers

In [ ]:
variaveis_numericas = df.select_dtypes(include=['int64', 'float64']).columns
variaveis_numericas = variaveis_numericas.drop('Person_ID')
f"Temos {len(variaveis_numericas)} variaveis numéricas: {', '.join(variaveis_numericas)}"

In [ ]:
f, axes = plt.subplots(3, 3, figsize=(15, 10))

line = 0
column = 0
for i in variaveis_numericas:
    sns.boxplot(y=i, data=df, ax=axes[line][column])
    column += 1
    if column == 3:
        column = 0
        line += 1

# Modelo ML

## Pré processamento

In [180]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

In [181]:
df_train = df[['Daily_Screen_Time', 'Stress_Level', 'Anxiety', 'Physical_Activity', 'Occupation', 'Depression']]
df_train.head()

,Daily_Screen_Time,Stress_Level,Anxiety,Physical_Activity,Occupation,Depression
0,10.2,8,1,Low,Student,1
1,6.8,4,0,High,Student,0
2,5.5,10,0,Low,Employed,1
3,5.6,2,0,High,Employed,0
4,10.1,4,1,Low,Employed,1


In [182]:
# ohe = OneHotEncoder(sparse_output=False)
# df_train[ohe.get_feature_names_out(['Occupation'])] = ohe.fit_transform(df_train[['Occupation']])
# ordinal_encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
# df_train['Physical_Activity'] = ordinal_encoder.fit_transform(df_train[['Physical_Activity']])
# df_train.drop(['Occupation', 'Physical_Activity'], axis=1, inplace=True)
# df_train


In [183]:
ohe = OneHotEncoder(sparse_output=False)
df_train[ohe.get_feature_names_out()] = ohe.fit_transform(df_train[['Occupation', 'Physical_Activity']])
df_train.drop(['Occupation', 'Physical_Activity'], axis=1, inplace=True)
df_train

C:\Users\Andre Rafael\AppData\Local\Temp\ipykernel_13888\3379424259.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train[ohe.get_feature_names_out()] = ohe.fit_transform(df_train[['Occupation', 'Physical_Activity']])
C:\Users\Andre Rafael\AppData\Local\Temp\ipykernel_13888\3379424259.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train[ohe.get_feature_names_out()] = ohe.fit_transform(df_train[['Occupation', 'Physical_Activity']])
C:\Users\Andre Rafael\AppData\Local\Temp\ipykernel_13888\33794

,Daily_Screen_Time,Stress_Level,Anxiety,Depression,Occupation_Employed,Occupation_Student,Occupation_Unemployed,Physical_Activity_High,Physical_Activity_Low,Physical_Activity_Medium
0,10.2,8,1,1,0.0,1.0,0.0,0.0,1.0,0.0
1,6.8,4,0,0,0.0,1.0,0.0,1.0,0.0,0.0
2,5.5,10,0,1,1.0,0.0,0.0,0.0,1.0,0.0
3,5.6,2,0,0,1.0,0.0,0.0,1.0,0.0,0.0
4,10.1,4,1,1,1.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...
1995,4.8,1,0,1,1.0,0.0,0.0,0.0,0.0,1.0
1996,2.7,9,1,1,1.0,0.0,0.0,0.0,1.0,0.0
1997,2.6,10,1,0,1.0,0.0,0.0,0.0,0.0,1.0
1998,3.2,9,0,0,0.0,1.0,0.0,1.0,0.0,0.0


In [184]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_train.drop('Depression', axis=1))
df_scaled

array([[ 1.13018574,  0.89810455,  0.98905984, ..., -0.69863813,
         1.39016234, -0.70339769],
       [-0.04967563, -0.48866179, -1.01106117, ...,  1.43135617,
        -0.71934045, -0.70339769],
       [-0.5007991 ,  1.59148773, -1.01106117, ..., -0.69863813,
         1.39016234, -0.70339769],
       ...,
       [-1.50715145,  1.59148773,  0.98905984, ..., -0.69863813,
        -0.71934045,  1.42167086],
       [-1.29894061,  1.24479614, -1.01106117, ...,  1.43135617,
        -0.71934045, -0.70339769],
       [ 0.29734242,  0.55141297, -1.01106117, ..., -0.69863813,
         1.39016234, -0.70339769]], shape=(2000, 9))

In [195]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [186]:
X_train, X_test, y_train, y_test = train_test_split(df_scaled, df_train['Depression'], test_size=0.2, random_state=42)

In [204]:
svc_model = SVC(kernel='linear')
svc_model.fit(X_train, y_train)
result = svc_model.predict(X_test)
print(classification_report(y_test, result))
confusion_matrix(y_test, result)

              precision    recall  f1-score   support

           0       0.58      0.58      0.58       198
           1       0.59      0.59      0.59       202

    accuracy                           0.58       400
   macro avg       0.58      0.58      0.58       400
weighted avg       0.58      0.58      0.58       400



array([[114,  84],
       [ 83, 119]])

In [ ]:
logistic_regression_model = LogisticRegression()
logistic_regression_model.fit(X_train, y_train)
print(classification_report(
    y_test, 
    logistic_regression_model.predict(X_test)
))


              precision    recall  f1-score   support

           0       0.58      0.59      0.58       198
           1       0.59      0.59      0.59       202

    accuracy                           0.59       400
   macro avg       0.59      0.59      0.59       400
weighted avg       0.59      0.59      0.59       400



In [205]:
tree = DecisionTreeClassifier(criterion='gini', random_state=42)
tree.fit(X_train, y_train)
result = tree.predict(X_test)
print(classification_report(
    y_test, 
    result
))
confusion_matrix(y_test, result)

              precision    recall  f1-score   support

           0       0.54      0.59      0.56       198
           1       0.56      0.51      0.53       202

    accuracy                           0.55       400
   macro avg       0.55      0.55      0.55       400
weighted avg       0.55      0.55      0.55       400



array([[116,  82],
       [ 99, 103]])

In [202]:
neighbors_model = KNeighborsClassifier(n_neighbors=5)
neighbors_model.fit(X_train, y_train)
print(classification_report(
    y_test,
    neighbors_model.predict(X_test)
))

              precision    recall  f1-score   support

           0       0.56      0.60      0.58       198
           1       0.58      0.54      0.56       202

    accuracy                           0.57       400
   macro avg       0.57      0.57      0.57       400
weighted avg       0.57      0.57      0.57       400



In [203]:
gaussian_process_model = GaussianProcessClassifier()
gaussian_process_model.fit(X_train, y_train)
gaussian_process_model.predict(X_test)
print(
    classification_report(
        y_test,
        gaussian_process_model.predict(X_test)
    )
)

              precision    recall  f1-score   support

           0       0.59      0.60      0.59       198
           1       0.60      0.59      0.59       202

    accuracy                           0.59       400
   macro avg       0.59      0.59      0.59       400
weighted avg       0.59      0.59      0.59       400

